In [6]:
%pip install pandas numpy scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import numpy as np

train = pd.read_csv('data/solicitudes_train.csv')
test  = pd.read_csv('data/solicitudes_test.csv')
banco = pd.read_csv('data/comportamiento_bancario.csv')

In [8]:
# Juntamos train y test para calcular las features igual en los dos
solicitudes = pd.concat([train, test])[['ID_Solicitud', 'ID_Usuario', 'Fecha_Solicitud']]

# Cada solicitud se cruza con todos los meses de su usuario
hist = solicitudes.merge(banco, on='ID_Usuario', how='left')

# Calculamos cuántos meses antes de la solicitud es cada registro de comportamiento bancario
fecha_sol = pd.to_datetime(hist['Fecha_Solicitud'])
fecha_mes = pd.to_datetime(hist['Mes_Referencia']) #mes de cada registro de comportamiento bancario
hist['meses_atras'] = (fecha_sol.dt.year - fecha_mes.dt.year) * 12 + (fecha_sol.dt.month - fecha_mes.dt.month)

# Usamos solo los 12 meses anteriores a la solicitud
VENTANA = 12
hist = hist[(hist['meses_atras'] >= 1) & (hist['meses_atras'] <= VENTANA)]

In [9]:
hist['tuvo_atraso_30'] = hist['Dias_Atraso_Otras_Deudas'] > 30

feat = hist.groupby('ID_Solicitud').agg(
    saldo_prom      = ('Saldo_Promedio_Mensual', 'mean'),
    saldo_min       = ('Saldo_Promedio_Mensual', 'min'),
    saldo_std       = ('Saldo_Promedio_Mensual', 'std'),
    atraso_max      = ('Dias_Atraso_Otras_Deudas', 'max'),
    atraso_prom     = ('Dias_Atraso_Otras_Deudas', 'mean'),
    meses_atraso_30 = ('tuvo_atraso_30', 'sum'),   # cuántos meses debió más de 30 días
    meses_informados= ('Mes_Referencia', 'count'), # cuántos meses de registro tenés
)

# Filtramos por los ultimos 3 meses para evaluar el comportamiento reciente
ult3 = hist[hist['meses_atras'] <= 3]
feat_ult3 = ult3.groupby('ID_Solicitud').agg(
    saldo_prom_ult3 = ('Saldo_Promedio_Mensual', 'mean'),
    atraso_max_ult3 = ('Dias_Atraso_Otras_Deudas', 'max'),
)

# Unimos las features de los ultimos 3 meses con las features generales
feat = feat.join(feat_ult3)
feat['tendencia_saldo'] = feat['saldo_prom_ult3'] / feat['saldo_prom']  # >1 = viene mejorando
feat = feat.reset_index()

In [10]:
# Unimos las features con los datasets de train y test
train_f = train.merge(feat, on='ID_Solicitud', how='left')
test_f  = test.merge(feat,  on='ID_Solicitud', how='left')

print(train_f.shape, test_f.shape)
print(train_f.isna().sum()[lambda s: s > 0])

train_f.groupby('Estado_Final')[['atraso_max', 'meses_atraso_30', 'saldo_prom']].mean()

(32416, 18) (3061, 17)
saldo_prom_ult3    4
atraso_max_ult3    4
tendencia_saldo    4
dtype: int64


,atraso_max,meses_atraso_30,saldo_prom
Estado_Final,,,
0,45.567202,1.511700,2731.242914
1,31.285594,0.769059,5892.787649


In [11]:
# Verificamos si a partir de sus sueldo, puede pagar la cuota del préstamo y si su saldo promedio es mayor que su sueldo
for df in (train_f, test_f):
    df['cuota_sobre_ingreso'] = df['Monto_Solicitado'] / df['Ingreso_Mensual']   
    df['saldo_sobre_ingreso'] = df['saldo_prom'] / df['Ingreso_Mensual']

# Motivo_Prestamo es texto: lo pasamos a categoría, con las mismas categorías en train y test
motivos = sorted(train_f['Motivo_Prestamo'].unique())
for df in (train_f, test_f):
    df['Motivo_Prestamo'] = pd.Categorical(df['Motivo_Prestamo'], categories=motivos)

FEATURES = ['Monto_Solicitado', 'Tasa_Interes', 'Ingreso_Mensual', 'Motivo_Prestamo',
            'saldo_prom', 'saldo_min', 'saldo_std', 'atraso_max', 'atraso_prom', 'meses_atraso_30',
            'meses_informados', 'saldo_prom_ult3', 'atraso_max_ult3', 'tendencia_saldo',
            'cuota_sobre_ingreso', 'saldo_sobre_ingreso']

# Datasets de train y test para el modelo 
X      = train_f[FEATURES]
y      = train_f['Estado_Final']
grupos = train_f['ID_Usuario']
X_test = test_f[FEATURES]

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# Dividimos 80% para entrenar, 20% para validar
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Entrenamos un modelo de Gradient Boosting con HistGradientBoostingClassifier
modelo = HistGradientBoostingClassifier(categorical_features='from_dtype', random_state=42)
modelo.fit(X_train, y_train)

# Probabilidad de que pague (columna 1 de predict_proba)
p_val = modelo.predict_proba(X_val)[:, 1]

# AUC mide que tan bien el modelo ordena a los clientes
print('AUC validación:', round(roc_auc_score(y_val, p_val), 4))

AUC validación: 0.838


In [13]:
PRESUPUESTO = 2_000_000

# Creamos un dataset con la información de validación y la probabilidad de pago
val = train_f.loc[X_val.index].copy()
val['p_pago'] = p_val

# Esperanza matemática de cada préstamo:
# con prob. p gano monto*tasa; con prob. (1-p) pierdo el monto
val['ganancia_esperada'] = (val['p_pago'] * val['Monto_Solicitado'] * val['Tasa_Interes']
                            - (1 - val['p_pago']) * val['Monto_Solicitado'])

# Ganancia esperada por cada dólar prestado
val['rentabilidad'] = val['ganancia_esperada'] / val['Monto_Solicitado']

# Ordenamos de más a menos rentable y vamos sumando montos
val = val.sort_values('rentabilidad', ascending=False)
val['acumulado'] = val['Monto_Solicitado'].cumsum()

# Aprobamos si conviene (esperanza > 0) y si todavía entra en el presupuesto
val['aprobado'] = ((val['ganancia_esperada'] > 0) & (val['acumulado'] <= PRESUPUESTO)).astype(int)

# Vemos qué tan bien le fue al modelo comparando con lo que pasó en realidad (Estado_Final)
val['resultado'] = np.where(val['Estado_Final'] == 1,
                            val['Monto_Solicitado'] * val['Tasa_Interes'],
                            -val['Monto_Solicitado'])

# Calculamos cuántos préstamos aprobados hubo, cuánto se prestó y cuánto se ganó realmente
aprobados = val[val['aprobado'] == 1]
print('Préstamos aprobados:', len(aprobados))
print('Monto prestado:', aprobados['Monto_Solicitado'].sum())
print('Ganancia real:', round(aprobados['resultado'].sum()))

Préstamos aprobados: 73
Monto prestado: 1987000
Ganancia real: 478581


In [14]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest no acepta categorías ni NaN: los convertimos a números
def evaluar(val, p_val, ordenar_por='rentabilidad'):
    """Aplica nuestra regla de decisión y devuelve la ganancia REAL obtenida."""
    val = val.copy()
    val['p_pago'] = p_val
    val['ganancia_esperada'] = (val['p_pago'] * val['Monto_Solicitado'] * val['Tasa_Interes']
                                - (1 - val['p_pago']) * val['Monto_Solicitado'])
    val['rentabilidad'] = val['ganancia_esperada'] / val['Monto_Solicitado']
    val = val.sort_values(ordenar_por, ascending=False)
    val['acumulado'] = val['Monto_Solicitado'].cumsum()
    val['aprobado'] = ((val['ganancia_esperada'] > 0) & (val['acumulado'] <= PRESUPUESTO)).astype(int)
    val['resultado'] = np.where(val['Estado_Final'] == 1,
                                val['Monto_Solicitado'] * val['Tasa_Interes'],
                                -val['Monto_Solicitado'])
    return val.loc[val['aprobado'] == 1, 'resultado'].sum()

# Función para preparar los datos para Random Forest
def preparar_rf(Z):
    """Random Forest no acepta categorías ni NaN: los convertimos a números."""
    Z = Z.copy()
    Z['Motivo_Prestamo'] = Z['Motivo_Prestamo'].cat.codes
    return Z.fillna(0)

# Entrenamos varios modelos y comparamos resultados
resultados = {'HGB orden rentabilidad': [], 'HGB orden ganancia': [], 'HGB orden prob. pago': [],
              'HGB regularizado': [], 'Random Forest': []}

for semilla in range(5):
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=semilla)
    
    chicos = X_val['Monto_Solicitado'] <= 11000
    X_val = X_val[chicos]
    val = train_f.loc[X_val.index]

    m = HistGradientBoostingClassifier(categorical_features='from_dtype', random_state=42).fit(X_train, y_train)
    p = m.predict_proba(X_val)[:, 1]
    resultados['HGB orden rentabilidad'].append(evaluar(val, p))
    resultados['HGB orden ganancia'].append(evaluar(val, p, 'ganancia_esperada'))
    resultados['HGB orden prob. pago'].append(evaluar(val, p, 'p_pago'))

    m = HistGradientBoostingClassifier(categorical_features='from_dtype', max_depth=4, min_samples_leaf=50,
                                       learning_rate=0.05, max_iter=400, random_state=42).fit(X_train, y_train)
    resultados['HGB regularizado'].append(evaluar(val, m.predict_proba(X_val)[:, 1]))

    m = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42).fit(preparar_rf(X_train), y_train)
    resultados['Random Forest'].append(evaluar(val, m.predict_proba(preparar_rf(X_val))[:, 1]))

# Mostramos los resultados promedio
pd.DataFrame(resultados).mean().round().sort_values(ascending=False)

HGB regularizado          530909.0
HGB orden rentabilidad    530571.0
Random Forest             516895.0
HGB orden ganancia        481498.0
HGB orden prob. pago      467625.0
dtype: float64

In [15]:
modelo_final = HistGradientBoostingClassifier(categorical_features='from_dtype', random_state=42)
modelo_final.fit(X, y)

,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""Categorical"" and ""Enum"" are considered to be categorical features. The input must be a dataframe that is supported by narwhals (or supports it): :func:`narwhals.from_native` must work. This is the case, for instance, for pandas and polars DataFrames.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing val

In [16]:
modelo_final = HistGradientBoostingClassifier(
    categorical_features='from_dtype',
    max_depth=4,            # árboles más cortos
    min_samples_leaf=50,    # cada hoja necesita al menos 50 clientes
    learning_rate=0.05,
    max_iter=400,
    random_state=42,
)
modelo_final.fit(X, y)

,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.05
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",400
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",4
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",50
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""Categorical"" and ""Enum"" are considered to be categorical features. The input must be a dataframe that is supported by narwhals (or supports it): :func:`narwhals.from_native` must work. This is the case, for instance, for pandas and polars DataFrames.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing value

In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Función para preparar los datos para regresión logística
def preparar_lr(Z):
    """La regresión logística necesita todo numérico y sin NaN."""
    Z = pd.get_dummies(Z, columns=['Motivo_Prestamo'])   # one-hot: una columna 0/1 por motivo
    return Z.fillna(0)

In [20]:
modelo_final = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
modelo_final.fit(preparar_lr(X), y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](22,)","['Monto_Solicitado','Tasa_Interes','Ingreso_Mensual',..., 'Motivo_Prestamo_Salud','Motivo_Prestamo_Vehiculo', 'Motivo_Prestamo_Vivienda']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,22
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [17]:
# Creamos un dataset con la información de test y la probabilidad de pago
sub = test_f[['ID_Solicitud', 'Monto_Solicitado', 'Tasa_Interes']].copy()
sub['p_pago'] = modelo_final.predict_proba(X_test)[:, 1]

sub['ganancia_esperada'] = (sub['p_pago'] * sub['Monto_Solicitado'] * sub['Tasa_Interes']
                            - (1 - sub['p_pago']) * sub['Monto_Solicitado'])
sub['rentabilidad'] = sub['ganancia_esperada'] / sub['Monto_Solicitado']

# Ordenamos por rentabilidad y vamos sumando montos hasta llegar al presupuesto
sub = sub.sort_values('rentabilidad', ascending=False)
sub['acumulado'] = sub['Monto_Solicitado'].cumsum()
sub['aprobado'] = ((sub['ganancia_esperada'] > 0) & (sub['acumulado'] <= PRESUPUESTO)).astype(int)

# Controlamos no pasarnos del presupuesto
monto_total = sub.loc[sub['aprobado'] == 1, 'Monto_Solicitado'].sum()
print('Aprobados:', sub['aprobado'].sum(), 'de', len(sub))
print(f'Monto total aprobado: $ {monto_total:,.0f}')
assert monto_total <= PRESUPUESTO

sub = sub.sort_values('ID_Solicitud')
sub = sub.rename(columns={'aprobado': 'Decision_Aprobacion'})
sub[['ID_Solicitud', 'Decision_Aprobacion']].to_csv('submissions/entrega_02.csv', index=False)

Aprobados: 653 de 3061
Monto total aprobado: $ 1,997,600


In [18]:
# Creamos un dataset con la información de test y la probabilidad de pago
sub = test_f[['ID_Solicitud', 'Monto_Solicitado', 'Tasa_Interes']].copy()
sub['p_pago'] = modelo_final.predict_proba(X_test)[:, 1]

sub['ganancia_esperada'] = (sub['p_pago'] * sub['Monto_Solicitado'] * sub['Tasa_Interes']
                            - (1 - sub['p_pago']) * sub['Monto_Solicitado'])
sub['rentabilidad'] = sub['ganancia_esperada'] / sub['Monto_Solicitado']

# Ordenamos de más a menos rentable
sub = sub.sort_values('rentabilidad', ascending=False)

# Recorremos uno por uno: aprobamos si conviene y si todavía entra en lo que queda del presupuesto
restante = PRESUPUESTO
aprobado = []
for monto, ganancia in zip(sub['Monto_Solicitado'], sub['ganancia_esperada']):
    if ganancia > 0 and monto <= restante:
        aprobado.append(1)
        restante -= monto
    else:
        aprobado.append(0)
sub['aprobado'] = aprobado

# Controlamos no pasarnos del presupuesto
monto_total = sub.loc[sub['aprobado'] == 1, 'Monto_Solicitado'].sum()
print('Aprobados:', sub['aprobado'].sum(), 'de', len(sub))
print(f'Monto total aprobado: $ {monto_total:,.0f}')
print(f'Presupuesto sin usar: $ {restante:,.0f}')
assert monto_total <= PRESUPUESTO

sub = sub.sort_values('ID_Solicitud')
sub = sub.rename(columns={'aprobado': 'Decision_Aprobacion'})
sub[['ID_Solicitud', 'Decision_Aprobacion']].to_csv('submissions/entrega_03.csv', index=False)

Aprobados: 656 de 3061
Monto total aprobado: $ 1,999,800
Presupuesto sin usar: $ 200


In [21]:
# Creamos un dataset con la información de test y la probabilidad de pago
sub = test_f[['ID_Solicitud', 'Monto_Solicitado', 'Tasa_Interes']].copy()
sub['p_pago'] = modelo_final.predict_proba(preparar_lr(X_test))[:, 1]

sub['ganancia_esperada'] = (sub['p_pago'] * sub['Monto_Solicitado'] * sub['Tasa_Interes']
                            - (1 - sub['p_pago']) * sub['Monto_Solicitado'])
sub['rentabilidad'] = sub['ganancia_esperada'] / sub['Monto_Solicitado']

# Ordenamos de más a menos rentable
sub = sub.sort_values('rentabilidad', ascending=False)

# Recorremos uno por uno: aprobamos si conviene y si todavía entra en lo que queda del presupuesto
restante = PRESUPUESTO
aprobado = []
for monto, ganancia in zip(sub['Monto_Solicitado'], sub['ganancia_esperada']):
    if ganancia > 0 and monto <= restante:
        aprobado.append(1)
        restante -= monto
    else:
        aprobado.append(0)
sub['aprobado'] = aprobado

# Controlamos no pasarnos del presupuesto
monto_total = sub.loc[sub['aprobado'] == 1, 'Monto_Solicitado'].sum()
print('Aprobados:', sub['aprobado'].sum(), 'de', len(sub))
print(f'Monto total aprobado: $ {monto_total:,.0f}')
print(f'Presupuesto sin usar: $ {restante:,.0f}')
assert monto_total <= PRESUPUESTO

sub = sub.sort_values('ID_Solicitud')
sub = sub.rename(columns={'aprobado': 'Decision_Aprobacion'})
sub[['ID_Solicitud', 'Decision_Aprobacion']].to_csv('submissions/entrega_04.csv', index=False)

Aprobados: 610 de 3061
Monto total aprobado: $ 1,999,600
Presupuesto sin usar: $ 400


In [22]:
modelo_hgb = HistGradientBoostingClassifier(categorical_features='from_dtype', max_depth=4, min_samples_leaf=50,
                                            learning_rate=0.05, max_iter=400, random_state=42)
modelo_hgb.fit(X, y)

modelo_lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
modelo_lr.fit(preparar_lr(X), y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](22,)","['Monto_Solicitado','Tasa_Interes','Ingreso_Mensual',..., 'Motivo_Prestamo_Salud','Motivo_Prestamo_Vehiculo', 'Motivo_Prestamo_Vivienda']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,22
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [23]:
def generar_entrega(p_pago, nombre_archivo):
    """Aplica nuestra regla de decisión sobre test y guarda el CSV para Kaggle."""
    sub = test_f[['ID_Solicitud', 'Monto_Solicitado', 'Tasa_Interes']].copy()
    sub['p_pago'] = p_pago
    sub['ganancia_esperada'] = (sub['p_pago'] * sub['Monto_Solicitado'] * sub['Tasa_Interes']
                                - (1 - sub['p_pago']) * sub['Monto_Solicitado'])
    sub['rentabilidad'] = sub['ganancia_esperada'] / sub['Monto_Solicitado']

    # Ordenamos de más a menos rentable
    sub = sub.sort_values('rentabilidad', ascending=False)

    # Recorremos uno por uno: aprobamos si conviene y si todavía entra en lo que queda del presupuesto
    restante = PRESUPUESTO
    aprobado = []
    for monto, ganancia in zip(sub['Monto_Solicitado'], sub['ganancia_esperada']):
        if ganancia > 0 and monto <= restante:
            aprobado.append(1)
            restante -= monto
        else:
            aprobado.append(0)
    sub['aprobado'] = aprobado

    # Controlamos no pasarnos del presupuesto
    monto_total = sub.loc[sub['aprobado'] == 1, 'Monto_Solicitado'].sum()
    assert monto_total <= PRESUPUESTO
    print(f"{nombre_archivo}: {sub['aprobado'].sum()} aprobados, $ {monto_total:,.0f} prestados, $ {restante:,.0f} sin usar")

    sub = sub.sort_values('ID_Solicitud')
    sub = sub.rename(columns={'aprobado': 'Decision_Aprobacion'})
    sub[['ID_Solicitud', 'Decision_Aprobacion']].to_csv('submissions/' + nombre_archivo, index=False)

In [24]:
p_hgb = modelo_hgb.predict_proba(X_test)[:, 1]
p_lr  = modelo_lr.predict_proba(preparar_lr(X_test))[:, 1]

generar_entrega(p_hgb, 'entrega_03.csv')                  # gradient boosting regularizado
generar_entrega(p_lr, 'entrega_04.csv')                   # regresión logística
generar_entrega((p_hgb + p_lr) / 2, 'entrega_05.csv')     # promedio de los dos (ensamble)

entrega_03.csv: 656 aprobados, $ 1,999,800 prestados, $ 200 sin usar
entrega_04.csv: 610 aprobados, $ 1,999,600 prestados, $ 400 sin usar
entrega_05.csv: 637 aprobados, $ 2,000,000 prestados, $ 0 sin usar
